# Goodgorithm — Quality classifier training

Trains the classifier #238's research found as the strongest quality signal in the ranking pipeline: a TF-IDF + logistic regression classifier predicting whether a post is `genuinely-uplifting-and-substantive` (AUC 0.7734 on the labelled set). See `CLAUDE.md`'s Architecture section and the #238 epic for full context.

**What this notebook does:** loads a fresh training export (see `doc/LABEL_EXPORT.md` for how to produce one from the `labeling` schema), trains a TF-IDF + logistic regression classifier, **re-assesses the deployment threshold from scratch every run** (a mandatory, blocking step — never hardcoded), exports to ONNX, and uploads to the `goodgorithm-models` R2 bucket under a `quality-classifier/` prefix.

**This doesn't need a GPU** — TF-IDF + logistic regression trains on CPU in minutes.

**Before running:** produce a fresh export via `doc/LABEL_EXPORT.md` and have the resulting JSON file ready to upload below. This notebook also fetches `processing/src/util/text_normalize.py` from a pinned commit, so normalization can't drift between training and inference — update `TEXT_NORMALIZE_COMMIT` below if that file has changed since the last training run.

In [ ]:
!pip install -q scikit-learn skl2onnx onnxruntime boto3

## Load the training export

The canonical, committed dataset lives at `training/data/feed-quality-labels.json` (see `doc/LABEL_EXPORT.md` for how it's produced and kept up to date). Upload that file below, or a fresher export produced the same way if you've collected more labels since the committed version was last updated — an array of objects with at least `id`, `text`, and `category` fields (`category` is one of `genuinely-uplifting-and-substantive` / `thin-but-harmless` / `shouldnt-be-shown`). Extra columns (`source`, `hashtags`, `attachments`, `quote_content`, `original_created_at`, `rank_score`, `batch`) are fine and ignored here.

If you're running this notebook with local repo access rather than in Colab/Kaggle, you can skip the upload widget entirely and just point `DATA_PATH` at `../training/data/feed-quality-labels.json` directly.

In [ ]:
DATA_PATH = "training_export.json"

try:
    from google.colab import files
    uploaded = files.upload()
    DATA_PATH = next(iter(uploaded))
except ImportError:
    pass  # not running in Colab -- point DATA_PATH at a local file instead (e.g. ../training/data/feed-quality-labels.json)

import json

with open(DATA_PATH) as f:
    rows = json.load(f)

print(f"loaded {len(rows)} rows from {DATA_PATH}")
assert rows, "training export is empty"
assert all("id" in r and "text" in r and "category" in r for r in rows), "every row needs id + text + category"

## Fetch the shared text-normalization file

In [ ]:
TEXT_NORMALIZE_COMMIT = "a45ca7363bc23531a313c8901ec93ff0bbd137f9"  # bump if processing/src/util/text_normalize.py has changed since

import urllib.request

url = (
    f"https://raw.githubusercontent.com/goodgorithm/goodgorithm/"
    f"{TEXT_NORMALIZE_COMMIT}/processing/src/util/text_normalize.py"
)
urllib.request.urlretrieve(url, "text_normalize.py")

import text_normalize

print("fetched text_normalize.py @", TEXT_NORMALIZE_COMMIT)

## Build the training target

Binary target: `genuinely-uplifting-and-substantive` vs. everything else (`thin-but-harmless` + `shouldnt-be-shown` collapsed together) — matching #243's original labeling scheme, not a 3-way classification. `thin-but-harmless` stays in the negative class during *training* (it's the harder, more informative negative to separate from), but every *reported* AUC below is measured against `shouldnt-be-shown` only — the established convention throughout #238/#243's research (see `signal_diagnostics_238.py`'s `continuous_signal_report`), so numbers here stay directly comparable to everything already posted on those issues.

In [ ]:
GOOD = "genuinely-uplifting-and-substantive"
BAD = "shouldnt-be-shown"

import numpy as np

texts = [text_normalize.normalize_text(r["text"]) for r in rows]
y = np.array([1 if r["category"] == GOOD else 0 for r in rows])

print(f"{int(y.sum())} positive / {len(y) - int(y.sum())} negative ({y.mean():.1%} positive rate)")

## Vectorize + train

`TfidfVectorizer(stop_words="english")` — #243's feature-engineering research (see that issue's closing comments for the full numbers) found this plain config beats every alternative tried for *this* classifier specifically: bigrams hurt (-0.0058 AUC), blending in extra engineered features hurt (-0.0325), and min_df/max_df filtering only ever hurt, monotonically, the more aggressive it got. The full, unfiltered, single-word vocabulary is doing real work. **Do not** copy the political classifier's `ngram_range`/`sublinear_tf`/`min_df`/`stop_words=None` choices — those were tuned for a different classifier and a different research question.

The one thing shared with the political classifier: `token_pattern=r"\b\w+\b"`, overriding sklearn's default (`(?u)\b\w\w+\b`) — an inline regex flag (`(?u)`) skl2onnx's RE2-based exported tokenizer doesn't support, a real ONNX-export correctness issue independent of the stopword question. This specific combination (English stopword removal + the ONNX-safe token pattern) hasn't been validated end-to-end before this notebook — the parity check in the Export section below is what actually confirms it works, not this note.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = TfidfVectorizer(stop_words="english", token_pattern=r"\b\w+\b")
X = vectorizer.fit_transform(texts)
print("vocab size:", len(vectorizer.vocabulary_))

clf = LogisticRegression(class_weight="balanced", max_iter=1000)

## Honest evaluation (out-of-fold)

Same 5-fold `StratifiedKFold(random_state=238243)` + `cross_val_predict` harness every #243/#242 research script used, so this run's numbers stay directly comparable to the research trail. These out-of-fold probabilities — never the full-fit model's own scores, which would overstate separation — are what feed the threshold re-assessment section below.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=238243)
oof_proba = cross_val_predict(clf, X, y, cv=skf, method="predict_proba")[:, 1]


def auc_separation(pos, neg):
    """Rank-based AUC (Mann-Whitney U): P(a random positive-group score >
    a random negative-group score), ties counted as 0.5. Matches
    the rank-based AUC computation used throughout #238's research,
    ported here so this notebook has no dependency on non-committed
    research scripts."""
    n_pos, n_neg = len(pos), len(neg)
    combined = sorted([(s, "pos") for s in pos] + [(s, "neg") for s in neg], key=lambda t: t[0])
    ranks = {}
    i = 0
    while i < len(combined):
        j = i
        while j < len(combined) and combined[j][0] == combined[i][0]:
            j += 1
        avg_rank = (i + 1 + j) / 2.0
        for k in range(i, j):
            ranks[k] = avg_rank
        i = j
    rank_sum_pos = sum(ranks[idx] for idx, (_, label) in enumerate(combined) if label == "pos")
    u_pos = rank_sum_pos - n_pos * (n_pos + 1) / 2.0
    return u_pos / (n_pos * n_neg)


pos_scores = [p for r, p in zip(rows, oof_proba) if r["category"] == GOOD]
neg_scores = [p for r, p in zip(rows, oof_proba) if r["category"] == BAD]
auc = auc_separation(pos_scores, neg_scores)
print(f"out-of-fold AUC (good vs shouldnt-be-shown): {auc:.4f}")
print("(first-run reference: 0.7734 -- a large deviation here means something about this export/normalization differs from the original research pipeline, worth investigating before trusting this run)")

## Full-set fit

The actual artifact — trained on every row. The out-of-fold scores above (not this model's own predictions on its own training data) are what the threshold re-assessment below uses.

In [ ]:
clf.fit(X, y)
print("trained on full set:", X.shape[0], "posts")

## Threshold re-assessment — mandatory every run

**This is not a one-time decision frozen at some number.** The deployment threshold (0.39 as of the first run of this process, decided in a dedicated #238 measurement — see that issue's closing comments for the full derivation) is re-derived from scratch every training run, against this run's own out-of-fold scores — the same methodology originally established for that measurement, ported here so it's inseparable from training rather than a separate script someone has to remember to also run.

**Every cell below this section is gated behind an explicit `THRESHOLD` value you set after reading the tables this section prints.** There is no formula that picks it for you, on purpose — the same "operationalize the good-retention constraint, don't just optimize separation" discipline #238 used the first time this was decided.

In [ ]:
import random


def category_kept_rates(all_results, kept_results):
    kept_ids = {r["id"] for r in kept_results}
    out = {"overall": len(kept_results) / len(all_results) if all_results else 0.0}
    categories = {r["category"] for r in all_results}
    for cat in categories:
        cat_all = [r for r in all_results if r["category"] == cat]
        cat_kept = sum(1 for r in cat_all if r["id"] in kept_ids)
        out[cat] = cat_kept / len(cat_all) if cat_all else 0.0
    return out


def sweep(results, thresholds):
    n_bad_total = sum(1 for r in results if r["category"] == BAD)
    out = []
    for th in thresholds:
        kept = [r for r in results if r["score"] >= th]
        excluded = [r for r in results if r["score"] < th]
        rates = category_kept_rates(results, kept)
        n_bad_excluded = sum(1 for r in excluded if r["category"] == BAD)
        precision = n_bad_excluded / len(excluded) if excluded else None
        recall = n_bad_excluded / n_bad_total if n_bad_total else None
        f1 = (2 * precision * recall / (precision + recall)) if precision and recall and (precision + recall) > 0 else 0.0
        good_kept = rates.get(GOOD, 0.0)
        bad_kept = rates.get(BAD, 0.0)
        out.append({
            "threshold": round(th, 4), "overall_kept": rates["overall"], "good_kept": good_kept,
            "thin_kept": rates.get("thin-but-harmless", 0.0), "bad_kept": bad_kept,
            "exclusion_precision": precision, "exclusion_recall": recall, "exclusion_f1": f1,
            "youden_j": good_kept - bad_kept,
        })
    return out


score_results = [{"id": r["id"], "category": r["category"], "score": s} for r, s in zip(rows, oof_proba)]
thresholds = [round(i * 0.01, 2) for i in range(0, 101)]
sweep_rows = sweep(score_results, thresholds)

best_j = max(sweep_rows, key=lambda r: r["youden_j"])
best_f1 = max(sweep_rows, key=lambda r: r["exclusion_f1"])
print(f"max Youden's J:   threshold={best_j['threshold']:.2f}  J={best_j['youden_j']:.4f}  good={best_j['good_kept']:.1%}  bad={best_j['bad_kept']:.1%}  overall={best_j['overall_kept']:.1%}")
print(f"max exclusion F1: threshold={best_f1['threshold']:.2f}  F1={best_f1['exclusion_f1']:.4f}  good={best_f1['good_kept']:.1%}  bad={best_f1['bad_kept']:.1%}  overall={best_f1['overall_kept']:.1%}")

print("\ngood-retention-floor-constrained candidates:")
print(f"{'floor':<8}| {'threshold':>9} | {'good_kept':>10} | {'bad_kept':>9} | {'overall_kept':>13}")
for floor in (0.95, 0.90, 0.85, 0.80, 0.70):
    eligible = [r for r in sweep_rows if r["good_kept"] >= floor]
    if not eligible:
        print(f"{floor:<8.0%}| no threshold clears this floor")
        continue
    best = max(eligible, key=lambda r: r["threshold"])
    print(f"{floor:<8.0%}| {best['threshold']:>9.2f} | {best['good_kept']:>9.1%} | {best['bad_kept']:>8.1%} | {best['overall_kept']:>12.1%}")

print("\nbootstrap stability check (Youden's J argmax, 500 resamples):")
rng = random.Random(238244)
n = len(score_results)
boot_thresholds = []
for _ in range(500):
    sample = [score_results[rng.randrange(n)] for _ in range(n)]
    boot_best = max(sweep(sample, thresholds), key=lambda r: r["youden_j"])
    boot_thresholds.append(boot_best["threshold"])
boot_thresholds.sort()
median = boot_thresholds[len(boot_thresholds) // 2]
p5 = boot_thresholds[int(len(boot_thresholds) * 0.05)]
p95 = boot_thresholds[int(len(boot_thresholds) * 0.95)]
print(f"median={median:.2f}  [p5={p5:.2f}, p95={p95:.2f}]  (point estimate was {best_j['threshold']:.2f})")

For direct comparison against what's currently live, check the previous published version's `config.json` (`training/r2_release.py --model quality current`, then fetch that version's `config.json` from R2 — e.g. via the R2/Supabase dashboard, or `boto3` once the R2 credentials cell below has run). Not automated in this cell since this notebook has no repo/CLI access when run in Colab.

In [ ]:
THRESHOLD = None  # <-- set this after reading the tables above

assert THRESHOLD is not None, "review the threshold-selection tables above and set THRESHOLD explicitly before continuing -- every cell below this one depends on it"
print(f"THRESHOLD = {THRESHOLD}")

## Spot-check

Hand-written examples — a model acing the metrics above but failing obvious spot-checks is a red flag worth catching before publishing. Replace/extend with real production examples pulled from staging when available.

In [ ]:
spot_checks = [
    ("A local shelter just found homes for 40 rescue dogs this weekend through a new fostering program.", "expect high -- substantive"),
    ("happy friday everyone! \U0001F389", "expect low -- thin greeting"),
    ("New research shows how urban gardens are helping restore pollinator populations in cities.", "expect high -- substantive"),
    ("good morning \U0001F60A", "expect low -- thin greeting"),
    ("Scientists just mapped a previously unknown coral reef ecosystem using AI-assisted imaging.", "expect high -- substantive"),
]
for text, note in spot_checks:
    vec = vectorizer.transform([text_normalize.normalize_text(text)])
    prob = clf.predict_proba(vec)[0][1]
    kept = "KEEP" if prob >= THRESHOLD else "cut "
    print(f"[{kept}] ({prob:.2f}) {text}  -- {note}")

## Export to ONNX

Verify numerically against `clf.predict_proba()`, and assert the exported graph's output name/width match expectations — same discipline as the political classifier's export cell. This classifier doesn't use `sublinear_tf` (unlike the political classifier), so the specific known numerical gap that config produces doesn't apply here — but the parity check stays as the general safety net regardless, since `token_pattern`'s RE2 conversion is untested territory for this exact vectorizer config until this cell actually runs.

In [ ]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import StringTensorType
from sklearn.pipeline import Pipeline
import onnxruntime as ort

pipeline = Pipeline([("tfidf", vectorizer), ("clf", clf)])
onnx_model = convert_sklearn(
    pipeline,
    initial_types=[("input", StringTensorType([None, 1]))],
    options={id(clf): {"zipmap": False}},
)
onnx_bytes = onnx_model.SerializeToString()

sess = ort.InferenceSession(onnx_bytes, providers=["CPUExecutionProvider"])
spot_check_texts = [text for text, _ in spot_checks]
sample = np.array([[text_normalize.normalize_text(t)] for t in spot_check_texts], dtype=object)
onnx_out = sess.run(None, {"input": sample})
output_names = [o.name for o in sess.get_outputs()]
assert output_names[1] == "probabilities", f"unexpected ONNX output order: {output_names}"
onnx_probs = onnx_out[1]
sklearn_probs = clf.predict_proba(vectorizer.transform([text_normalize.normalize_text(t) for t in spot_check_texts]))
# atol=0.01, tighter than the political classifier's 0.03 -- no sublinear_tf here, so there's
# no known/accepted numerical gap to absorb; a tighter tolerance is strictly
# more likely to catch a real token_pattern-related mismatch if one exists.
assert np.allclose(onnx_probs, sklearn_probs, atol=0.01), "ONNX/sklearn parity check failed"
assert onnx_probs.shape[1] == 2, "ONNX output width should be 2 (binary classifier)"
print("ONNX export verified: parity OK, output width 2")

with open("model.onnx", "wb") as f:
    f.write(onnx_bytes)

## Package `config.json`

In [ ]:
from datetime import datetime, timezone

VERSION = "v1"

config = {
    "version": VERSION,
    "text_normalize_source_commit": TEXT_NORMALIZE_COMMIT,
    "labels": ["other", "genuinely_uplifting_and_substantive"],
    "threshold": THRESHOLD,
    "tfidf": {"stop_words": "english", "token_pattern": r"\b\w+\b"},
    "dataset_composition": {
        "source": "labeling schema export (see doc/LABEL_EXPORT.md)",
        "n_posts": len(rows),
        "n_positive": int(y.sum()),
        "n_negative": int(len(y) - y.sum()),
    },
    "out_of_fold_auc_good_vs_bad": auc,
    "threshold_selection": {
        "sweep": sweep_rows,
        "best_youden_j": best_j,
        "best_exclusion_f1": best_f1,
        "bootstrap_median": median,
        "bootstrap_p5": p5,
        "bootstrap_p95": p95,
    },
    "trained_at": datetime.now(timezone.utc).isoformat(),
}

with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

print(json.dumps({k: v for k, v in config.items() if k != "threshold_selection"}, indent=2))
print("\n(threshold_selection's full sweep table omitted from this printout for brevity -- it is in config.json)")

## Upload to R2

Uploads always happen unconditionally — versioned artifacts are cheap and safe to publish; promoting a version to production is a separate, deliberate step (see below). Only two artifacts here — no separate `vocab.json`-equivalent, the TF-IDF vocabulary is baked into the exported ONNX model.

In [ ]:
R2_MODELS_ACCOUNT_ID = R2_MODELS_ACCESS_KEY_ID = R2_MODELS_SECRET_ACCESS_KEY = R2_MODELS_BUCKET_NAME = None

try:
    from google.colab import userdata

    R2_MODELS_ACCOUNT_ID = userdata.get("R2_MODELS_ACCOUNT_ID")
    R2_MODELS_ACCESS_KEY_ID = userdata.get("R2_MODELS_ACCESS_KEY_ID")
    R2_MODELS_SECRET_ACCESS_KEY = userdata.get("R2_MODELS_SECRET_ACCESS_KEY")
    R2_MODELS_BUCKET_NAME = userdata.get("R2_MODELS_BUCKET_NAME")
except Exception as e:
    print(f"Colab secrets unavailable ({type(e).__name__}: {e}), trying Kaggle secrets...")

if not R2_MODELS_ACCOUNT_ID:
    try:
        from kaggle_secrets import UserSecretsClient

        secrets = UserSecretsClient()
        R2_MODELS_ACCOUNT_ID = secrets.get_secret("R2_MODELS_ACCOUNT_ID")
        R2_MODELS_ACCESS_KEY_ID = secrets.get_secret("R2_MODELS_ACCESS_KEY_ID")
        R2_MODELS_SECRET_ACCESS_KEY = secrets.get_secret("R2_MODELS_SECRET_ACCESS_KEY")
        R2_MODELS_BUCKET_NAME = secrets.get_secret("R2_MODELS_BUCKET_NAME")
    except Exception as e:
        print(f"Kaggle secrets unavailable ({type(e).__name__}: {e}), falling back to manual values...")

if not R2_MODELS_ACCOUNT_ID:
    # Manual fallback -- fill these in locally, never commit real values.
    R2_MODELS_ACCOUNT_ID = ""
    R2_MODELS_ACCESS_KEY_ID = ""
    R2_MODELS_SECRET_ACCESS_KEY = ""
    R2_MODELS_BUCKET_NAME = ""

assert (
    R2_MODELS_ACCOUNT_ID
    and R2_MODELS_ACCESS_KEY_ID
    and R2_MODELS_SECRET_ACCESS_KEY
    and R2_MODELS_BUCKET_NAME
), "R2 credentials not set -- see the markdown cell above"

In [ ]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url=f"https://{R2_MODELS_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_MODELS_ACCESS_KEY_ID,
    aws_secret_access_key=R2_MODELS_SECRET_ACCESS_KEY,
    region_name="auto",
)

PREFIX = f"quality-classifier/{VERSION}"
s3.upload_file("model.onnx", R2_MODELS_BUCKET_NAME, f"{PREFIX}/model.onnx")
s3.upload_file("config.json", R2_MODELS_BUCKET_NAME, f"{PREFIX}/config.json")
print(f"uploaded to s3://{R2_MODELS_BUCKET_NAME}/{PREFIX}/")

## Promote to production

Deliberately not done from this notebook — no `gh`/repo access from Colab. After checking the eval numbers, spot-checks, and the threshold you settled on above, promote from a machine with an authenticated `gh` CLI:

```
cd training && uv run python r2_release.py --model quality publish v1
```

See the `release-quality-classifier` skill for the full checklist.